# 02. Data + Model inferencing: как получать LayerIORecord

В этом ноутбуке:

- выбираем модель по YAML-имени;
- подаём батч PIL-изображений;
- получаем `LayerIORecord` по всем `nn.Linear`;
- анализируем слои, размерности, статистики;
- показываем влияние `hook_filter` и `limits.max_records_per_layer`.


In [ ]:
import torch

MODEL_YAML_NAME = "clip_vit_b32"
DATASET_YAML_NAME = "flickr30k"
DATA_ROOT = "./data_tutorials"
IMAGE_DIR = "./images"  # если директории нет, используем raw dataset
N_IMAGES = 8
DEVICE_OVERRIDE = "cuda" if torch.cuda.is_available() else "cpu"

print("MODEL_YAML_NAME:", MODEL_YAML_NAME)
print("DATASET_YAML_NAME:", DATASET_YAML_NAME)
print("IMAGE_DIR:", IMAGE_DIR)
print("DEVICE_OVERRIDE:", DEVICE_OVERRIDE)


In [ ]:
from __future__ import annotations

import os
import sys
import platform
import time
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import torch
from PIL import Image
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from dataset.data_raw.providers.hf import register_all_adapters
from dataset.data_raw.providers.hf.auth import get_hf_token
from dataset.data_raw.registry import create_dataset
from dataset.models.registry import create_model
from dataset.shared.collector_service import CollectorService
from dataset.shared.shared_dataset import SharedModelDataset
from dataset.shared.compatibility_index import CompatibilityIndex


def load_hydra_cfg(config_path: str = "conf/config.yaml", overrides: list[str] | None = None):
    cfg_path = repo_root / config_path
    with initialize_config_dir(version_base=None, config_dir=str(cfg_path.parent.resolve())):
        cfg = compose(config_name=cfg_path.stem, overrides=overrides or [])
    return cfg


def resolve_dataset_cfg(cfg, dataset_yaml_name: str):
    data_cfg = cfg.data
    config_dirs = list(data_cfg.get("dataset_config_dirs", ["conf/data/datasets"]))
    for item in config_dirs:
        candidate = repo_root / str(item) / f"{dataset_yaml_name}.yaml"
        if candidate.exists():
            ds_cfg = OmegaConf.load(candidate)
            override_map = cfg.data.get("dataset_overrides", {})
            if dataset_yaml_name in override_map:
                ds_cfg = OmegaConf.merge(ds_cfg, override_map[dataset_yaml_name])
            return ds_cfg
    raise FileNotFoundError(f"Не найден YAML датасета: {dataset_yaml_name}")


def resolve_model_cfg(cfg, model_yaml_name: str):
    models_cfg = cfg.models if "models" in cfg else {}
    config_dirs = list(models_cfg.get("model_config_dirs", ["conf/data/models"]))
    for item in config_dirs:
        candidate = repo_root / str(item) / f"{model_yaml_name}.yaml"
        if candidate.exists():
            return OmegaConf.load(candidate)
    raise FileNotFoundError(f"Не найден YAML модели: {model_yaml_name}")


def show_image_grid(pil_list, n: int = 16, title: str | None = None):
    if not pil_list:
        print("Список изображений пуст")
        return
    images = pil_list[:n]
    cols = min(4, len(images))
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]

    idx = 0
    for r in range(rows):
        for c in range(cols):
            ax = axes[r][c]
            ax.axis("off")
            if idx < len(images):
                ax.imshow(images[idx])
                ax.set_title(f"#{idx}")
            idx += 1
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def print_yaml_section(path_to_yaml: str):
    path = repo_root / path_to_yaml
    if not path.exists():
        print(f"Файл не найден: {path}")
        return
    print("=" * 80)
    print(f"RAW YAML: {path}")
    print("=" * 80)
    print(path.read_text(encoding="utf-8"))
    print("=" * 80)
    print("OmegaConf (resolve=False)")
    print("=" * 80)
    obj = OmegaConf.load(path)
    print(OmegaConf.to_yaml(obj, resolve=False))
    print("=" * 80)
    print("OmegaConf (resolve=True, если возможно)")
    print("=" * 80)
    try:
        print(OmegaConf.to_yaml(obj, resolve=True))
    except Exception as exc:
        print(f"Не удалось resolve=True: {exc}")


def ensure_hf_token(cfg):
    token = get_hf_token(cfg)
    if not token:
        raise ValueError(
            "HF token missing in top-level config: set hf.token. "
            "Для gated ресурсов также примите лицензию на странице HF."
        )
    return token


def print_system_info():
    print(f"Python: {platform.python_version()}")
    print(f"Torch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"CUDA device count: {torch.cuda.device_count()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")


In [ ]:
print_system_info()
print("repo_root:", repo_root)


## Quick start (опционально одной ячейкой)

Поставьте `QUICK_START = True`, чтобы быстро выполнить минимальный сценарий: raw batch -> model.run -> summary.


In [ ]:
QUICK_START = False

if QUICK_START:
    cfg_qs = load_hydra_cfg(overrides=[f"data.path={DATA_ROOT}"])
    m_cfg_qs = resolve_model_cfg(cfg_qs, MODEL_YAML_NAME)
    m_cfg_qs_dict = OmegaConf.to_container(m_cfg_qs, resolve=True)
    m_cfg_qs_dict["device"] = DEVICE_OVERRIDE
    model_qs = create_model(MODEL_YAML_NAME, m_cfg_qs_dict, OmegaConf.to_container(cfg_qs, resolve=True))
    model_qs.load()

    register_all_adapters()
    ds_cfg_qs = resolve_dataset_cfg(load_hydra_cfg(overrides=[
        f"data.path={DATA_ROOT}",
        f"data.enabled_datasets=[{DATASET_YAML_NAME}]",
    ]), DATASET_YAML_NAME)
    ds_qs = create_dataset(
        name=DATASET_YAML_NAME,
        cfg=OmegaConf.to_container(ds_cfg_qs, resolve=True),
        global_root=str(DATA_ROOT),
        seed=42,
        hf_cfg=OmegaConf.to_container(cfg_qs.hf, resolve=True),
    )
    ds_qs.start()
    batch_qs = [s.image for s in ds_qs.get_batch(min(8, N_IMAGES))]
    records_qs = model_qs.run(batch_qs)

    print("records:", len(records_qs))
    if records_qs:
        print("first:", records_qs[0].layer_name, records_qs[0].inputs.shape, records_qs[0].outputs.shape)

    ds_qs.close()
    model_qs.unload()


## Схема YAML модели

Ниже показываем шаблон `conf/data/models/_model_.yaml` и реальный конфиг модели.

| Ключ | Что означает | Как влияет |
|---|---|---|
| `hf_repo` | модель на HF Hub | источник весов/процессора |
| `cache_subdir` | локальный кэш под `data.path` | повторное использование весов |
| `batch_size` | размер инференс-job | объём данных на шаг |
| `dtype` | float16/float32 | скорость, VRAM, стабильность |
| `run_mode` | vision_only/encoder_only/full | какая часть модели считается |
| `hook_filter` | include/exclude regex | какие `Linear` слои хукаются |
| `limits.max_records_per_layer` | ограничение N строк | защита от OOM/перегруза |


In [ ]:
print_yaml_section("conf/data/models/_model_.yaml")
print_yaml_section(f"conf/data/models/{MODEL_YAML_NAME}.yaml")


In [ ]:
cfg = load_hydra_cfg(overrides=[f"data.path={DATA_ROOT}"])
model_cfg = resolve_model_cfg(cfg, MODEL_YAML_NAME)

if bool(model_cfg.get("gated", False)):
    ensure_hf_token(cfg)

model_cfg_runtime = OmegaConf.to_container(model_cfg, resolve=True)
model_cfg_runtime["device"] = DEVICE_OVERRIDE

global_cfg_dict = OmegaConf.to_container(cfg, resolve=True)
model = create_model(MODEL_YAML_NAME, model_cfg_runtime, global_cfg_dict)
model.load()
print("model.stats():")
print(model.stats())


## Два способа собрать PIL-батч

### A) Из локальной директории


In [ ]:
def load_pil_from_dir(image_dir: str, n: int = 8):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    p = Path(image_dir)
    if not p.exists():
        return []

    files = [x for x in sorted(p.rglob("*")) if x.suffix.lower() in exts]
    pil_list = []
    for fp in files[:n]:
        try:
            pil_list.append(Image.open(fp).convert("RGB"))
        except Exception:
            continue
    return pil_list

local_pil = load_pil_from_dir(IMAGE_DIR, n=N_IMAGES)
print("local_pil count:", len(local_pil))
if local_pil:
    show_image_grid(local_pil, n=min(len(local_pil), 16), title="Локальные изображения")


### B) Из raw датасета


In [ ]:
register_all_adapters()

dataset_cfg = resolve_dataset_cfg(
    load_hydra_cfg(overrides=[f"data.path={DATA_ROOT}", f"data.enabled_datasets=[{DATASET_YAML_NAME}]"]),
    DATASET_YAML_NAME,
)

if bool(dataset_cfg.get("gated", False)):
    ensure_hf_token(cfg)

raw_dataset = create_dataset(
    name=DATASET_YAML_NAME,
    cfg=OmegaConf.to_container(dataset_cfg, resolve=True),
    global_root=str(DATA_ROOT),
    seed=42,
    hf_cfg=OmegaConf.to_container(cfg.hf, resolve=True),
)
raw_dataset.start()

raw_samples = raw_dataset.get_batch(N_IMAGES)
raw_pil = [item.image for item in raw_samples]
print("raw_pil count:", len(raw_pil))
show_image_grid(raw_pil, n=min(len(raw_pil), 16), title=f"Raw dataset: {DATASET_YAML_NAME}")


## Quick start: выбираем локальные, если есть, иначе raw


In [ ]:
pil_batch = local_pil if len(local_pil) >= N_IMAGES else raw_pil
print("Используем batch из", "local_pil" if pil_batch is local_pil else "raw_pil", "размер:", len(pil_batch))


In [ ]:
layer_records = model.run(pil_batch)
print("Количество LayerIORecord:", len(layer_records))

summary_rows = []
for rec in layer_records:
    n_rows = int(rec.inputs.shape[0])
    d_in = int(rec.inputs.shape[1]) if rec.inputs.ndim == 2 and rec.inputs.shape[0] > 0 else 0
    d_out = int(rec.outputs.shape[1]) if rec.outputs.ndim == 2 and rec.outputs.shape[0] > 0 else 0
    summary_rows.append((rec.layer_name, n_rows, d_in, d_out, rec.inputs.dtype, rec.inputs.device.type))

summary_rows = sorted(summary_rows, key=lambda x: x[1], reverse=True)
print("Top-10 слоёв по числу строк N:")
for row in summary_rows[:10]:
    print(row)


In [ ]:
# Детальный просмотр одного слоя
if not layer_records:
    raise RuntimeError("Нет layer records. Проверьте модель и входной батч.")

chosen = layer_records[0]
print("layer:", chosen.layer_name)
print("inputs shape:", tuple(chosen.inputs.shape), chosen.inputs.dtype, chosen.inputs.device)
print("outputs shape:", tuple(chosen.outputs.shape), chosen.outputs.dtype, chosen.outputs.device)

x = chosen.inputs
print("x mean/std:", float(x.mean()), float(x.std()))
print("x p01/p50/p99:", [float(torch.quantile(x, q)) for q in [0.01, 0.5, 0.99]])


In [ ]:
# Визуализация: гистограмма одной размерности эмбеддинга
if chosen.inputs.shape[0] > 0 and chosen.inputs.shape[1] > 0:
    dim_idx = 0
    values = chosen.inputs[:, dim_idx].cpu().numpy()
    plt.figure(figsize=(6, 4))
    plt.hist(values, bins=40)
    plt.title(f"Histogram of input dim={dim_idx} ({chosen.layer_name})")
    plt.xlabel("value")
    plt.ylabel("count")
    plt.show()


## Демонстрация hook filtering


In [ ]:
filtered_cfg = OmegaConf.to_container(model_cfg, resolve=True)
filtered_cfg["device"] = DEVICE_OVERRIDE
filtered_cfg.setdefault("hook_filter", {})

# Для CLIP имеет смысл оставить только vision-модули
filtered_cfg["hook_filter"]["include_regex"] = r"^vision_model\."
filtered_cfg["hook_filter"]["exclude_regex"] = None

model_filtered = create_model(MODEL_YAML_NAME, filtered_cfg, global_cfg_dict)
model_filtered.load()
filtered_records = model_filtered.run(pil_batch)

print("Слоёв без фильтра:", len(layer_records))
print("Слоёв с include_regex=^vision_model\.:", len(filtered_records))


## Демонстрация limits.max_records_per_layer


In [ ]:
limited_cfg = OmegaConf.to_container(model_cfg, resolve=True)
limited_cfg["device"] = DEVICE_OVERRIDE
limited_cfg.setdefault("limits", {})
limited_cfg["limits"]["max_records_per_layer"] = 64

model_limited = create_model(MODEL_YAML_NAME, limited_cfg, global_cfg_dict)
model_limited.load()
limited_records = model_limited.run(pil_batch)

if limited_records:
    r0 = limited_records[0]
    print("Пример слоя:", r0.layer_name)
    print("rows inputs:", r0.inputs.shape[0], "rows outputs:", r0.outputs.shape[0])
    print("meta:", r0.meta)


## Troubleshooting

- **Проблемы скачивания модели**:
  - проверьте интернет, `hf.token`, права доступа к gated-моделям.
- **CUDA OOM**:
  - уменьшите `batch_size`, переключите `dtype=float32`/`float16` по ситуации.
- **Проблемы с процессором (size mismatch)**:
  - проверьте `preprocess.processor` и `run_mode`.
- **`trust_remote_code`**:
  - включайте только при реальной необходимости.


In [ ]:
# Cleanup
for obj_name in ["model", "model_filtered", "model_limited"]:
    try:
        globals()[obj_name].unload()
    except Exception:
        pass

try:
    raw_dataset.close()
except Exception:
    pass

print("Cleanup done")
